# Invocations API: Handoff MCP Scenario

This notebook mirrors `review_bot.scenarios.handoff_claims_exception_routing`. It teaches the `handoff-claims-exception-routing` scenario in small executable blocks, with an added focus on **MCP tool usage**: the custom invocation payload, scenario metadata, pattern anatomy, the flow diagram, the local MCP tool context, agent roles, the Responses-vs-Invocations API fit, workflow execution, and the Invocations API response contract.

## 1. Notebook Setup

Run this notebook from the project virtual environment after installing the package with `python -m pip install -e .`. The MCP tools run locally over stdio with no network access and no credentials.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start):
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "review_bot").exists():
            return candidate
        nested = candidate / "invocations-api-review-bot"
        if (nested / "src" / "review_bot").exists():
            return nested
    raise RuntimeError("Could not find invocations-api-review-bot project root.")

PROJECT_ROOT = find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

PROJECT_ROOT

## 2. Load The Scenario And Helpers

Each scenario has its own Python module with a single `SCENARIO` object. Notebook helpers keep repeated learning/display code out of the notebook cells.

In [ ]:
from review_bot.scenarios.handoff_claims_exception_routing import SCENARIO
from review_bot.scenarios import SCENARIOS_BY_ID, get_scenario
from review_bot.notebook_helpers import (
    agent_roster,
    default_ollama_options,
    invocation_reference,
    load_sample_payload,
    mcp_tool_context,
    pattern_anatomy,
    scenario_summary,
)

scenario = SCENARIO
assert SCENARIOS_BY_ID["handoff-claims-exception-routing"] is scenario
assert get_scenario("handoff-claims-exception-routing") is scenario

scenario_summary(scenario)

## 3. Pattern Anatomy

This is the orchestration behavior. The Invocations API boundary owns the request and response JSON, while the payload selects the scenario and the agents call local MCP tools underneath.

In [ ]:
pattern_anatomy(scenario)

## 4. Flow Diagram

This runtime Mermaid diagram shows the API boundary, orchestration pattern, the scenario agents, and **dashed links to each agent's MCP tools**. The helper returns the Mermaid source and displays an image.

In [ ]:
from review_bot.diagram_helpers import display_scenario_flow, scenario_flow_diagram

flow_diagram = display_scenario_flow(scenario)
flow_diagram.mermaid

## 5. MCP Tool Context

These agents are grounded by the bundled `enterprise-context` MCP server. It is a local FastMCP stdio server with embedded sample enterprise data: **no network, no credentials, no writes**. The cells below list the tools the server exposes, show which tools each agent is allowed to call, and call every tool directly through the helper layer so you can see the deterministic results before the agents use them.

In [ ]:
from review_bot.mcp_servers import enterprise_context

context = mcp_tool_context(scenario)
print("Server exposes:", enterprise_context.AVAILABLE_TOOLS)
print("Tools used in this scenario:", context["all_tools_used"])
context["tools_by_agent"]

In [ ]:
# Call every local MCP tool directly. These are plain in-process calls into the same
# functions the stdio server exposes, so they need no Ollama and no subprocess.
record = enterprise_context.lookup_enterprise_record("CLAIM-88120")
policies = enterprise_context.search_policy("claim exception fraud")
steps = enterprise_context.list_playbook_steps("claims-exception")
score = enterprise_context.calculate_priority_score(impact=4, urgency=4, scope=3)
decision = enterprise_context.create_decision_log_entry("CLAIM-88120 review", "approve")
{
    "record": record,
    "policy_matches": policies["match_count"],
    "playbook_steps": steps["step_count"],
    "priority": score,
    "decision_log": decision,
}

## 6. Agent Roster

The roster shows who participates, what each agent is instructed to do, and (via the MCP context above) which tools each agent may call.

In [ ]:
agent_roster(scenario)

## 7. Load And Validate The Invocation Payload

Invocations owns its JSON contract. This is the same sample payload you can POST to `/invocations`.

In [ ]:
from review_bot.models import build_review_prompt, parse_review_request

payload = load_sample_payload(PROJECT_ROOT, scenario)
request = parse_review_request(payload)
assert request.scenario == scenario.id
request

## 8. API Fit: Responses vs Invocations

This MCP scenario is mirrored in both sample projects, so it is a good place to make the API choice explicit.

- **Use the Responses API** when the caller is a chat-style client that sends OpenAI-style `input` and expects a conversational reply. The server picks one scenario at startup and every turn flows through the same orchestration. The MCP tools run server-side and stay invisible to the caller's contract.
- **Use the Invocations API** when the caller is a system (webhook, CI job, batch worker) that sends a structured job payload and expects an application-owned JSON response with metadata. The scenario is selected per request, which makes it easy to route different job types to different orchestration patterns.

In both cases the same local `enterprise-context` MCP server provides deterministic, credential-free tool grounding, so the orchestration logic is identical and only the API boundary differs.

## 9. Build And Run The Workflow

This calls Ollama through the same `run_review` function used by the Invocations handler. When the agents run, the framework launches the local MCP stdio server with `approval_mode="never_require"` and exposes only each agent's allowed tools.

In [ ]:
prompt = build_review_prompt(request)
print(prompt)

In [ ]:
from review_bot.workflows import run_review

ollama_options = default_ollama_options()
response = await run_review(request, **ollama_options)
response.to_dict()

## 10. Invocations API Boundary

Use this reference to compare the in-process workflow with the hosted `/invocations` endpoint. The MCP tools stay entirely server-side, so the response contract is unchanged.

In [ ]:
invocation_reference(scenario, request)

## 11. Experiments

Try a different record id in the MCP tool cell, change one agent's `mcp_tools`, adjust `constraints`, `artifacts`, `max_tokens`, or `temperature`, or edit an agent instruction in the scenario module. Re-run prompt construction and the live call to compare results.